# Baseline BM25 --- Notebook de apoio

**Disciplina:** Inteligência Artificial --- FACOM/UFMS --- 2026/1

Este notebook carrega o `corpus.jsonl` gerado pelo notebook de coleta (`coleta_arxiv.ipynb`), treina um índice **BM25** e devolve o *top-k* para uma consulta-exemplo. É o seu **baseline obrigatório**. A partir daqui, você só precisa adicionar pré-processamento mais cuidadoso, comparar com KNN/denso e implementar os módulos de aprofundamento.

**Atenção:** este código é propositalmente "mínimo viável". Você é avaliado também pelo **rigor das suas escolhas** (justificativa dos hiperparâmetros, pré-processamento, conexão com o paradigma probabilístico de Naïve Bayes). Não entregue isto como está --- evolua-o.

In [ ]:
!pip install rank_bm25 nltk pandas

In [1]:
import json
import re
from pathlib import Path

import pandas as pd
from rank_bm25 import BM25Okapi

# stopwords do NLTK (somente uma vez)
import nltk
try:
    from nltk.corpus import stopwords
    STOP = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    from nltk.corpus import stopwords
    STOP = set(stopwords.words("english"))

## 1. Carregamento do corpus

Ajuste o caminho se o seu `corpus.jsonl` estiver em outro lugar.

In [2]:
CORPUS_PATH = Path("../data/corpus.jsonl")  # ajuste se necessário

docs = []
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        docs.append(json.loads(line))

print(f"Documentos carregados: {len(docs)}")
docs[0]

Documentos carregados: 2699


{'arxiv_id': '1501.00335',
 'title': 'A Data Transparency Framework for Mobile Applications',
 'abstract': "In today's mobile application marketplace, the ability of consumers to make informed choices regarding their privacy is extremely limited. Consumers largely rely on privacy policies and app permission mechanisms, but these do an inadequate job of conveying how information will be collected, used, stored, and shared. Mobile application developers go largely unrewarded for making apps more privacy conscious as it is difficult to communicate these features to consumers while they are searching for a new app. This paper provides an overview of a framework designed to help consumers make informed choices, and an incentive mechanism to encourage app developers to implement it. This framework includes machine readable privacy policies encouraged by mobile app stores and enhanced by user software agents. Such a framework would provide the foundation required for more advanced forms of pr

## 2. Pré-processamento

Mínimo aceitável: *lower-casing*, remoção de pontuação, remoção de *stopwords*. Não usamos *stemming* aqui para não esconder discussão (você pode adicionar e comparar).

In [3]:
TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z\-]+")

def tokenize(text: str):
    text = text.lower()
    tokens = TOKEN_RE.findall(text)
    return [t for t in tokens if t not in STOP and len(t) > 2]

tokenize("Automated Testing of Flutter Mobile Applications and Dart Projects")

['automated',
 'testing',
 'flutter',
 'mobile',
 'applications',
 'dart',
 'projects']

In [4]:
# Texto indexado = título + abstract
texts = [(d["title"] + ". " + d["abstract"]) for d in docs]
tokenized_corpus = [tokenize(t) for t in texts]
print("Exemplo de tokens do doc 0:", tokenized_corpus[0][:20])

Exemplo de tokens do doc 0: ['data', 'transparency', 'framework', 'mobile', 'applications', 'today', 'mobile', 'application', 'marketplace', 'ability', 'consumers', 'make', 'informed', 'choices', 'regarding', 'privacy', 'extremely', 'limited', 'consumers', 'largely']


## 3. Índice BM25

Hiperparâmetros padrão do `rank_bm25`: $k_1=1.5$, $b=0.75$. Documente no seu relatório qual valor adotou e por quê. O módulo M4 (otimização) pode ajustá-los empiricamente.

In [5]:
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)
print("Vocabulário BM25:", len(bm25.idf), "termos")

Vocabulário BM25: 21364 termos


## 4. Função de busca

In [6]:
def search(query: str, k: int = 10):
    q_tokens = tokenize(query)
    scores = bm25.get_scores(q_tokens)
    top_idx = scores.argsort()[::-1][:k]
    return [(int(i), float(scores[i]), docs[i]) for i in top_idx]

In [7]:
query = "automated testing of flutter mobile applications"
results = search(query, k=10)

for rank, (idx, score, d) in enumerate(results, 1):
    print(f"{rank:>2}. [{score:6.2f}] {d['title']}")
    print(f"     {d['arxiv_id']} | {d.get('primary_category')} | {d.get('published','')[:10]}")
    print(f"     {d['abstract'][:200]}...")
    print()

 1. [ 16.41] Automated Assessment in Mobile Programming Courses: Leveraging GitHub Classroom and Flutter for Enhanced Student Outcomes
     2504.04230 | cs.SE | 2025-04-05
     The growing demand for skilled mobile developers has made mobile programming courses an essential component of computer science curricula. However, these courses face unique challenges due to the comp...

 2. [ 14.30] Streamlining Acceptance Test Generation for Mobile Applications Through Large Language Models: An Industrial Case Study
     2510.18861 | cs.SE | 2025-10-21
     Mobile acceptance testing remains a bottleneck in modern software development, particularly for cross-platform mobile development using frameworks like Flutter. While developers increasingly rely on a...

 3. [ 12.33] How do Developers Test Android Applications?
     1801.06268 | cs.SE | 2018-01-19
     Enabling fully automated testing of mobile applications has recently become an important topic of study for both researchers and practitio

## 5. Gerando um arquivo `runs/bm25.trec` para avaliação

Formato TREC tradicional: `qid Q0 doc_id rank score system`. Esse formato é aceito por `pytrec_eval` e `ir_measures`, e é o que o script `evaluate.py` (no diretório `eval/`) consome.

In [8]:
queries_path = Path("../eval/queries.tsv")  # qid<TAB>texto da query
runs_dir = Path("./runs"); runs_dir.mkdir(exist_ok=True)
run_path = runs_dir / "bm25.trec"

queries = pd.read_csv(queries_path, sep="\t", names=["qid", "text"])
with open(run_path, "w", encoding="utf-8") as f:
    for _, q in queries.iterrows():
        for rank, (idx, score, d) in enumerate(search(q["text"], k=100), 1):
            f.write(f"{q['qid']} Q0 {d['arxiv_id']} {rank} {score:.6f} bm25\n")

print("Run salva em:", run_path)
!head -n 5 {run_path}

Run salva em: runs\bm25.trec


'head' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


## 6. Próximos passos

1. Substitua o tokenizador por algo mais robusto (e.g., spaCy) e compare.
2. Implemente um segundo *retriever* (KNN + TF-IDF, ou KNN + *embeddings*) e gere `runs/knn.trec`.
3. Construa as `qrels` (Seção 2 do template em `eval/qrels_example.tsv`).
4. Use `eval/evaluate.py` para comparar.
5. Implemente o(s) módulo(s) de aprofundamento e gere mais arquivos de *run*.
6. Escreva tudo no relatório SBC. ✍️